In [ ]:
import numpy as np
import pandas as pd
from rpy2.robjects import r, conversion, pandas2ri
from helper_functions import normalize_household_data, dict_to_named_list

In [ ]:
pandas2ri.activate()
r.source('Simulator/Simulator.R')
model_r = r['simulate_and_reformat']

In [ ]:
def prior(batch_size: int) -> np.ndarray:
    param_batch = np.zeros((batch_size, 12))
    # alpha
    param_batch[:, 0] = np.random.uniform(0, 0.1, batch_size)
    # beta
    param_batch[:, 1] = np.random.uniform(0, 10, batch_size)
    # delta
    param_batch[:, 2] = np.random.uniform(-3, 3, batch_size)
    # mu_inf
    param_batch[:, 3:8] = np.exp(np.random.normal(0, 1, (batch_size, 5)))
    # mu_susc
    param_batch[:, 8:10] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    # mu_protect
    param_batch[:, 10:] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    return param_batch

test = prior(1).flatten()

In [ ]:
param_names = ['alpha', 'beta', 'delta', 
               'mu_inf_SC', 'mu_inf_SA', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA',
               'mu_susc_C', 'mu_susc_A', 
               'mu_protect_acq', 'mu_protect_transm']

In [ ]:
def simulator(params: np.ndarray,
              variant: str = "alpha",
              selection_procedure: str = "pedcov") -> np.ndarray:
    """
    Simulate data with given parameters and reformat it to a numpy array.
    :param params: parameters for the simulation
    :param variant: variant of the virus (alpha or omicron)
    :param selection_procedure: selection procedure for the households (pedcov or random)
    :return: simulated data as numpy array
    """
    # create dict from params and param_names
    par_dict = dict(zip(param_names, params))
    # update dict with fixed parameters
    par_dict.update({'variant': variant, 'selection_procedure': selection_procedure})

    # minimal_length should be the maximal length of the time series in the real data set
    if par_dict['variant'] == "alpha":
        minimal_length = 8
    elif par_dict['variant'] == "omicron":
        minimal_length = 9
    else:
        raise ValueError("Variant not supported. Must be 'alpha' or 'omicron'.")

    # simulate data
    sim_data_r = model_r(dict_to_named_list(par_dict))
    # convert to pandas dataframe
    sim_data_full = conversion.rpy2py(sim_data_r)
    # normalize data and return as numpy array
    #sim_data_norm = normalize_household_data(sim_data_full, minimal_length=minimal_length, use_one_hot_encoding=True)
    return sim_data_full
#model_py.__name__ = 'simulate_and_reformat'

In [ ]:
sim_data = simulator(test)

In [ ]:
sim_data[sim_data['id_hh_origin'] == "42"]

In [ ]:
sim_data['id_hh_origin'].unique()

In [ ]:
# extract the original household ids
sim_data['id_hh_origin'].nunique()

In [ ]:
# columns: date_sympt_norm, infect_status_norm, age_norm, protected
# rows: empty individuals in the beginning (households are of same size)
# last row: end_followup_norm with 1

In [ ]:
# load a presimulation using pickle
import pickle
with open('presimulations/presim_file_1.pkl', 'rb') as f:
    presim = pickle.load(f)
    
with open('presimulations/presim_file_2.pkl', 'rb') as f:
    presim_2 = pickle.load(f)

In [ ]:
presim[0]['prior_draws']#-presim_2[5]['prior_draws']

In [ ]:
len(presim)

In [ ]:
presim_2[0]['prior_draws']#[:, :, :, 0]*1000

In [ ]:
with open('valid_data.pickle', 'rb') as f:
    valid_data = pickle.load(f)

In [ ]:
valid_data['prior_draws']